# SA3 correlation heatmap: cultural diversity vs cervical screening participation

Each SA3 area is one data point. For every Census measure and every AIHW window we ask:
"do areas with a higher share of X tend to have higher or lower participation?"

Area-level only (ecological): it says nothing about individual women.
ANALYST-ONLY output: do not show named-area results to users.

The `# %%` lines make each block a runnable cell in VS Code; you can also copy
each block into its own Jupyter cell.

### 1. Setup and load data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv("../scripts/.env")                  # .env lives in scripts/
assert os.getenv("DATABASE_URL"), "DATABASE_URL not found: check the path to .env"
# Name the driver explicitly: newer SQLAlchemy defaults to 'psycopg' (v3),
# but this project uses psycopg2
url = os.getenv("DATABASE_URL").replace("postgres://", "postgresql://", 1)
engine = create_engine(url.replace("postgresql://", "postgresql+psycopg2://", 1))

# All windows, all areas: the exclusions are applied in step 2, where you can see them
linked = pd.read_sql(text("SELECT * FROM v_sa3_linked"), engine)

# Age structure, for the age control: share of women 25-74 who are aged 65-74 (G09)
# text() stops the driver reading the % in LIKE as a parameter placeholder
age = pd.read_sql(text("""
    SELECT c.geo_code,
           SUM(c.value) FILTER (WHERE v.source_column = 'F_Tot_65_74')
             / NULLIF(SUM(c.value) FILTER (WHERE v.source_column LIKE 'F\\_Tot\\_%'), 0) AS share_65_74
    FROM census_area_value c
    JOIN lookup_census_variable v USING (variable_code)
    WHERE v.table_code = 'G09'
    GROUP BY c.geo_code
"""), engine)

df = linked.merge(age, on="geo_code", how="left")
num_cols = ["rate_pct", "eligible_pop", "women_25_74", "pct_born_overseas",
            "pct_other_language", "pct_low_proficiency", "pct_birthplace_ns", "share_65_74"]
df[num_cols] = df[num_cols].astype(float)       # Postgres NUMERIC arrives as Decimal
print(df.shape, "rows (expect 1020 = 340 areas x 3 windows)")

MEASURES = {"pct_born_overseas": "Born overseas (women 25-74)",
            "pct_other_language": "Other language at home (females)",
            "pct_low_proficiency": "Low English proficiency (females)"}
WINDOWS = {2018: "2018-22", 2019: "2019-23", 2020: "2020-24"}

### 2. Analysis rules

In [ ]:
def main_sample(d):
    """Main analysis: drop suppressed, over-100%, and tiny areas (<1,000 women 25-74)."""
    return d[~d["is_suppressed"] & ~d["over_100"] & (d["women_25_74"] >= 1000)]

def all_areas(d):
    """Sensitivity 1: everything usable (only suppressed and over-100% removed)."""
    return d[~d["is_suppressed"] & ~d["over_100"]]

def stated_birthplace(d):
    """Sensitivity 2: main sample, also dropping areas with >10% birthplace not stated."""
    return main_sample(d)[lambda x: x["pct_birthplace_ns"] <= 10]

for name, rule in [("main", main_sample), ("all areas", all_areas), ("birthplace stated", stated_birthplace)]:
    print(f"{name:18} {len(rule(df[df.year_start == 2020]))} areas (2020-24 window)")

# Sanity check: shares must lie between 0 and 100. Tiny areas can break this (ABS adds small
# random noise to cells), which is why the main sample drops areas under 1,000 women.
main_latest = main_sample(df)[lambda d: d["year_start"] == 2020]
bad = main_latest[(main_latest[list(MEASURES)] < 0).any(axis=1) | (main_latest[list(MEASURES)] > 100).any(axis=1)]
print(f"\nAreas in the main sample with an impossible share: {len(bad)} (expect 0)")
if len(bad):
    print(bad[["geo_code", "geo_name", "women_25_74", *MEASURES]].to_string(index=False))

### 3. Correlation functions

In [ ]:
def spearman(x, y):
    rho, p = spearmanr(x, y)
    return rho, p

def partial_spearman(x, y, z):
    """Spearman correlation of x and y after removing the (rank) effect of z.
    Rank all three, regress the ranks of x and y on the rank of z, correlate the residuals."""
    rx, ry, rz = (pd.Series(v).rank().to_numpy() for v in (x, y, z))
    resid = lambda a: a - np.polyval(np.polyfit(rz, a, 1), rz)
    rho, p = pearsonr(resid(rx), resid(ry))     # p is approximate (one extra parameter used)
    return rho, p

def correlation_table(d):
    rows = []
    for year, label in WINDOWS.items():
        w = d[d["year_start"] == year].dropna(subset=["rate_pct", "share_65_74", *MEASURES])
        if len(w) < 30:                          # too few areas for a meaningful correlation
            print(f"  skipped {label}: only {len(w)} areas in this sample")
            continue
        for col, mlabel in MEASURES.items():
            rho, p = spearman(w[col], w["rate_pct"])
            prho, pp = partial_spearman(w[col], w["rate_pct"], w["share_65_74"])
            rows.append(dict(measure=mlabel, window=label, n=len(w),
                             rho=rho, p=p, rho_age_adjusted=prho, p_age_adjusted=pp))
    return pd.DataFrame(rows)

results = correlation_table(main_sample(df))
print(results.round(3).to_string(index=False))

### 4. THE HEATMAP: measures x windows, before and after the age control

In [ ]:
def heatmap(ax, table, value_col, title):
    grid = table.pivot(index="measure", columns="window", values=value_col)
    grid = grid.loc[list(MEASURES.values()), list(WINDOWS.values())]
    im = ax.imshow(grid.values, cmap="PuOr", vmin=-1, vmax=1)   # diverging, centred on 0
    ax.set_xticks(range(grid.shape[1]), grid.columns)
    ax.set_yticks(range(grid.shape[0]), grid.index)
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            ax.text(j, i, f"{grid.values[i, j]:+.2f}", ha="center", va="center", fontsize=11)
    ax.set_title(title, fontsize=11)
    return im

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
heatmap(axes[0], results, "rho", "Spearman rho with participation")
im = heatmap(axes[1], results, "rho_age_adjusted", "Same, controlling for age (share aged 65-74)")
axes[1].set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label="correlation (-1 to +1)")
n_latest = results.loc[results["window"] == "2020-24", "n"].iloc[0]
fig.suptitle(f"SA3 areas, main sample (n = {n_latest} in 2020-24). "
             "Area-level association only, not individual behaviour.", fontsize=10)
plt.show()

### 5. Sensitivity check: does the answer survive different exclusion rules? (2020-24)

In [ ]:
sens = []
for name, rule in [("main", main_sample), ("all areas", all_areas), ("birthplace stated", stated_birthplace)]:
    t = correlation_table(rule(df))
    if t.empty:
        continue
    t = t[t["window"] == "2020-24"].assign(sample=name)
    sens.append(t[["sample", "measure", "n", "rho", "rho_age_adjusted"]])
print(pd.concat(sens).round(3).to_string(index=False))

### 6. Are the three measures really separate? (correlations between the measures)

In [ ]:
m = main_sample(df)[lambda d: d["year_start"] == 2020][list(MEASURES)]
inter = m.corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
im = ax.imshow(inter.values, cmap="PuOr", vmin=-1, vmax=1)
labels = [MEASURES[c].split(" (")[0] for c in inter.columns]
ax.set_xticks(range(3), labels, rotation=20, ha="right")
ax.set_yticks(range(3), labels)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{inter.values[i, j]:+.2f}", ha="center", va="center")
ax.set_title("Spearman correlation between the Census measures")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

### 7. Scatter plots behind the numbers (2020-24, main sample)

In [ ]:
s = main_sample(df)[lambda d: d["year_start"] == 2020]
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for ax, (col, label) in zip(axes, MEASURES.items()):
    ax.scatter(s[col], s["rate_pct"], s=12, alpha=0.6)
    rho, _ = spearman(s[col], s["rate_pct"])
    ax.set_xlabel(f"{label} (%)")
    ax.set_title(f"rho = {rho:+.2f}", fontsize=10)
axes[0].set_ylabel("Participation rate 2020-24 (%)")
plt.show()

In [ ]:
# Group areas into 5 equal groups by share of women born overseas,
# then compare participation in each group (2020-24, main sample)
s = main_sample(df)[lambda d: d["year_start"] == 2020].copy()
s["group"] = pd.qcut(s["pct_born_overseas"], 5,
                     labels=["Lowest 20%", "2nd", "3rd", "4th", "Highest 20%"])

summary = s.groupby("group", observed=True).agg(
    areas=("rate_pct", "size"),
    born_overseas_range=("pct_born_overseas", lambda x: f"{x.min():.0f}–{x.max():.0f}%"),
    median_participation=("rate_pct", "median"),
    middle_half=("rate_pct", lambda x: f"{x.quantile(.25):.0f}–{x.quantile(.75):.0f}%"))
print(summary.round(1))

plt.figure(figsize=(7, 3.5))
plt.boxplot([g["rate_pct"] for _, g in s.groupby("group", observed=True)])
plt.xticks(range(1, len(summary) + 1), summary.index)
plt.ylabel("Participation 2020-24 (%)")
plt.xlabel("Areas grouped by share of women born overseas")
plt.show()

In [ ]:
# Median participation per group, for each window (stability check)
rows = []
for year in [2018, 2019, 2020]:
    w = main_sample(df)[lambda d: d["year_start"] == year].copy()
    w["group"] = pd.qcut(w["pct_born_overseas"], 5,
                         labels=["Lowest 20%", "2nd", "3rd", "4th", "Highest 20%"])
    medians = w.groupby("group", observed=True)["rate_pct"].median().round(1)
    medians.name = WINDOWS[year]
    rows.append(medians)

print(pd.concat(rows, axis=1))

In [ ]:
import pandas as pd

for col in ["pct_other_language", "pct_low_proficiency"]:
    sub = ok[[col, "rate_pct"]].dropna().copy()
    sub["group"] = pd.qcut(sub[col], 5, labels=["Lowest 20%", "2nd", "3rd", "4th", "Highest 20%"])
    summary = sub.groupby("group", observed=True).agg(
        areas=("rate_pct", "size"),
        range_lo=(col, "min"), range_hi=(col, "max"),
        median=("rate_pct", "median"),
        q1=("rate_pct", lambda s: s.quantile(0.25)),
        q3=("rate_pct", lambda s: s.quantile(0.75)),
    ).round(1)
    print(f"\n{col}\n{summary.to_string()}")